## Run test: Qwen/Qwen2.5-Omni-7B

Model page: https://huggingface.co/Qwen/Qwen2.5-Omni-7B

Source: Hugging Face model README Quickstart. Inference Providers snippets are intentionally ignored because they exercise hosted network APIs rather than local AMD GPU inference.

This notebook uses a text-only smoke test derived from the README API path so it does not download remote media. `return_audio=False` and `disable_talker()` avoid speech generation while still validating local model loading and generation on GPU.


In [ ]:
%pip install -U accelerate qwen-omni-utils soundfile


In [ ]:
# README-aligned local GPU inference smoke test
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

model_path = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(model_path, torch_dtype="auto").to("cuda")
model.disable_talker()
processor = Qwen2_5OmniProcessor.from_pretrained(model_path)

conversation = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group."}
        ],
    },
    {"role": "user", "content": [{"type": "text", "text": "Who are you?"}]},
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
inputs = processor(
    text=text,
    audio=audios,
    images=images,
    videos=videos,
    return_tensors="pt",
    padding=True,
    use_audio_in_video=False,
)
inputs = inputs.to(model.device).to(model.dtype)

with torch.no_grad():
    text_ids = model.generate(**inputs, use_audio_in_video=False, return_audio=False, max_new_tokens=32)

output = processor.batch_decode(text_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(output)
print("model devices:", sorted({str(p.device) for p in model.parameters()}))
